# Collision Dataset Exploration

My main goal is to develop a pipeline to evaluate the location data of Toronto Open Data's KSI Collision dataset. 

However, there's alot of variety in the dataset. In order to best evaluate each collision record, I must segment the collisions into categories, and come up with methods to assess the quality of each category of collision.

The focus of the this notebook is to determine how to segment the collisions.

In [2]:
import geopandas as gpd
import pandas as pd

In [3]:
address_pattern = r'^\d{1,5}\w?\s{0,2}\w+\s?\w+$'
offset_pattern = r'^\d.?\d*\s?[mM]?(etres)?\s?\w{1,5}'

In [4]:
collision_data = pd.read_csv("../data/raw/collisions.csv")
collision_data = collision_data[['collision_id', 'stname1', 'stname2', 'stname3']]
collision_data.head()

,collision_id,stname1,stname2,stname3
0,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
1,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
2,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
3,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
4,2006:893184,WOODBINE AVE,O CONNOR DR,NaN


We segment the collisions based on the values of `stname1`, `stname2`, and `stname3`.

According to the database documentation, `stname1` and `stname2` will typically take on the names of streets (eg. Wilson Ave), and `stname3` will include information about distance and direction from the intersection described in `stname1` and `stname2`, (eg. 80m West of).

However, once we take a look at the dataset, we can see that  `stname1` and `stname2` can also take on values representing addresses (eg. 20 Redgrave Dr). 

We must also watch out for any of `stname1`, `stname2`, and/or `stname3` being null.

In [5]:
collision_data[collision_data['stname1'].isnull()]

,collision_id,stname1,stname2,stname3


Nice to see that there are not collisions with `stname1` null. Makes my life easier. Now let's check how many of them are addresses.

In [6]:
collision_data[collision_data['stname1'].str.match(address_pattern, na=False)]

,collision_id,stname1,stname2,stname3
65,2006:884310,460 CORONATION DR,NaN,NaN
66,2006:884310,460 CORONATION DR,NaN,NaN
67,2006:884310,460 CORONATION DR,NaN,NaN
112,2006:888108,840 KENNEDY RD,NaN,NaN
113,2006:888108,840 KENNEDY RD,NaN,NaN
...,...,...,...,...
20417,2024:4001888169,60 CLEARVIEW HTS,NaN,1 m South of
20429,2026:6000022030,93 TRETHEWEY DR,93 TRETHEWEY DR,20 m East of
20438,2023:3002177373,690 GLENCAIRN AVE,NaN,NaN
20439,2023:3002177373,690 GLENCAIRN AVE,NaN,NaN


What about collisions that don't have addresses in `stname1`....

In [7]:
collision_data[~collision_data['stname1'].str.match(address_pattern, na=True)]

,collision_id,stname1,stname2,stname3
0,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
1,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
2,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
3,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
4,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
...,...,...,...,...
20450,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of
20451,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of
20452,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of
20453,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of


How many collisions have `stname1` not be an address and `stname2` not null?

In [8]:
collision_data[~collision_data['stname1'].str.match(address_pattern, na=True) & collision_data['stname2'].notnull()]

,collision_id,stname1,stname2,stname3
0,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
1,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
2,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
3,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
4,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
...,...,...,...,...
20450,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of
20451,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of
20452,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of
20453,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of


What if `stname1` is not an address and `stname2` is null?

In [9]:
collision_data[~collision_data['stname1'].str.match(address_pattern, na=True) & collision_data['stname2'].isnull()]

,collision_id,stname1,stname2,stname3
53,2006:886552,3367 DUNDAS ST W,NaN,NaN
54,2006:886552,3367 DUNDAS ST W,NaN,NaN
143,2006:893365,1302 KING ST W,NaN,NaN
144,2006:893365,1302 KING ST W,NaN,NaN
257,2006:894735,910 DUNDAS ST E,NaN,NaN
...,...,...,...,...
20434,2024:4002513613,650 LAWRENCE AVE W,NaN,NaN
20435,2024:4002513613,650 LAWRENCE AVE W,NaN,NaN
20442,2024:4001619827,144 BLOOR ST W,NaN,NaN
20446,2024:4002513613,650 LAWRENCE AVE W,NaN,NaN


These still seem to produce addresses. I'm going to (un)safely assume these to be address collisions.

Now we check to see how many collisions have `stname1` not as an address, and `stname2` is an address.

In [10]:
collision_data[~collision_data['stname1'].str.match(address_pattern, na=True) & collision_data['stname2'].str.match(address_pattern, na=False)]

,collision_id,stname1,stname2,stname3
4483,2009:1114016,MCCULLOCH AVE,1 AUTUMN AVE,NaN
4484,2009:1114016,MCCULLOCH AVE,1 AUTUMN AVE,NaN
4485,2009:1114016,MCCULLOCH AVE,1 AUTUMN AVE,NaN
4486,2009:1114016,MCCULLOCH AVE,1 AUTUMN AVE,NaN
4487,2009:1114016,MCCULLOCH AVE,1 AUTUMN AVE,NaN
8351,2025:5001780286,KIPLING AVE,2116 KIPLING AVE,1 m East of
12230,2015:5002192601,DUFFERIN ST,1 AUTUMN AVE,NaN
12231,2015:5002192601,DUFFERIN ST,1 AUTUMN AVE,NaN
17377,2021:1001924845,DISCO RD,120,5 m West of
17378,2021:1001924845,DISCO RD,120,5 m West of


We then check for how many collisions have `stname1` and `stname2` not be addresses (and both are not null).

In [11]:
collision_data[~collision_data['stname1'].str.match(address_pattern, na=True) & ~collision_data['stname2'].str.match(address_pattern, na=True)]

,collision_id,stname1,stname2,stname3
0,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
1,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
2,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
3,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
4,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
...,...,...,...,...
20450,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of
20451,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of
20452,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of
20453,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of


We now check to see how many collisions have `stname1` and `stname2` not be addresses, and `stname3` be null.

In [12]:
collision_data[~collision_data['stname1'].str.match(address_pattern, na=True) & ~collision_data['stname2'].str.match(address_pattern, na=True) & collision_data['stname3'].isnull()]

,collision_id,stname1,stname2,stname3
0,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
1,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
2,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
3,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
4,2006:893184,WOODBINE AVE,O CONNOR DR,NaN
...,...,...,...,...
20410,2025:5000209670,FAYWOOD BLVD,WILSON AVE,NaN
20411,2025:5002507368,MCCOWAN RD,NUGGET AVE,NaN
20415,2026:6000251229,DUNDAS ST W,ACORN AVE,NaN
20431,2026:6000251229,DUNDAS ST W,ACORN AVE,NaN


How many collisions have `stname1`, and `stname2` be non-null and not address, with `stname3` not null?

In [13]:
collision_data[~collision_data['stname1'].str.match(address_pattern, na=True) & ~collision_data['stname2'].str.match(address_pattern, na=True) & collision_data['stname3'].notnull()]

,collision_id,stname1,stname2,stname3
13,2006:884090,BATHURST ST,DUNDAS ST W,60 NORTH OF
14,2006:884090,BATHURST ST,DUNDAS ST W,60 NORTH OF
5369,2023:4000062087,WOLFE AVE,COMMONWEALTH AVE,5 m West of
5425,2024:4001738447,LAWRENCE AVE E,DON MILLS RD,40 m West of
5462,2024:4001738447,LAWRENCE AVE E,DON MILLS RD,40 m West of
...,...,...,...,...
20450,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of
20451,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of
20452,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of
20453,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of


How many collisions have `stname1`, and `stname2` be non-null and not addresses, with `stname3` be an offset description?

In [14]:
collision_data[~collision_data['stname1'].str.match(address_pattern, na=False) & ~collision_data['stname2'].str.match(address_pattern, na=True) & collision_data['stname3'].str.match(offset_pattern, na=False)]

,collision_id,stname1,stname2,stname3
13,2006:884090,BATHURST ST,DUNDAS ST W,60 NORTH OF
14,2006:884090,BATHURST ST,DUNDAS ST W,60 NORTH OF
5369,2023:4000062087,WOLFE AVE,COMMONWEALTH AVE,5 m West of
5425,2024:4001738447,LAWRENCE AVE E,DON MILLS RD,40 m West of
5462,2024:4001738447,LAWRENCE AVE E,DON MILLS RD,40 m West of
...,...,...,...,...
20450,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of
20451,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of
20452,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of
20453,2025:5000721646,LAKE SHORE BLVD W,FIFTH ST,5 m East of


How many collisions have `stname1` and `stname2` be non-null and not addresses, while `stname3` is non-null and not an offset?

In [15]:
collision_data[~collision_data['stname1'].str.match(address_pattern, na=False) & ~collision_data['stname2'].str.match(address_pattern, na=True) & ~collision_data['stname3'].str.match(offset_pattern, na=True)]

,collision_id,stname1,stname2,stname3
13498,2017:7000701050,HALFMOON SQ,CANMORE BLVD,north of
13499,2017:7000701050,HALFMOON SQ,CANMORE BLVD,north of
13528,2017:7000795019,MORNINGSIDE AVE,SEWELLS RD,north of
13529,2017:7000795019,MORNINGSIDE AVE,SEWELLS RD,north of
13530,2017:7000795019,MORNINGSIDE AVE,SEWELLS RD,north of
13531,2017:7000795019,MORNINGSIDE AVE,SEWELLS RD,north of
13533,2017:7000795019,MORNINGSIDE AVE,SEWELLS RD,north of
14259,2018:8000041197,STEELES AVE E,EASTVALE DR,E of
14260,2018:8000041197,STEELES AVE E,EASTVALE DR,E of
14261,2018:8000053420,EGLINTON AVE E,ROSEMOUNT DR,W of


These collisions will have to be treated as intersection collisions.

This is where I close out my data exploration. It's certainly not perfect. There's definitely a bunch of collisions that have been miscategorized. But its better to have a pipeline built out first.

My exploration has led me to believe that there are two major categories of collisions:
1. Address collisions -> Any collision where `stname1` or `stname2` are addresses.
2. Intersection collisions -> Any collision that isn't an address collision

I agree that the definition of Intersection collision here seems a bit too simple, but if `stname1` and `stname2` both aren't addresses, then they have to be streets. And the intersection of two streets is....an intersection.

(We could try to investigate whether or not the two streets listed intersect but I don't wanna go down this rabbit hole any more than I already have.)

In [29]:
address_boundaries = pd.read_csv("../data/raw/property-boundaries.csv")
address_boundaries.head()

# Clean the address boundaries data by removing rows with missing or invalid addresses
address_boundaries = address_boundaries[address_boundaries['ADDRESS_NUMBER'].notnull() & address_boundaries['LINEAR_NAME_FULL'].notnull()]

In [30]:
address_boundaries.head()

,_id,PARCELID,FEATURE_TYPE,DATE_EFFECTIVE,DATE_EXPIRY,STATEDAREA,ADDRESS_NUMBER,LINEAR_NAME_FULL,OBJECTID,TRANS_ID_CREATE,TRANS_ID_EXPIRE,geometry
0,1,5090819,COMMON,2016-06-15,3000-01-01,17366.998291 sq.m,5000,Jane St,1,1,-1,"{""coordinates"": [[[[-79.5230957815308, 43.7743..."
1,2,5093788,COMMON,2016-06-15,3000-01-01,557.2834473 sq.m,343,Hullmar Dr,2,1,-1,"{""coordinates"": [[[[-79.525917669714, 43.77311..."
4,5,5124997,COMMON,2016-06-15,3000-01-01,18629.3165283 sq.m,40,Millwick Dr,5,1,-1,"{""coordinates"": [[[[-79.5579982280959, 43.7589..."
5,6,5094544,COMMON,2016-06-15,3000-01-01,557.322876 sq.m,24,Wheelwright Cres,6,1,-1,"{""coordinates"": [[[[-79.5256074247922, 43.7724..."
6,7,5095105,COMMON,2016-06-15,3000-01-01,592.6890869 sq.m,36,Wheelwright Cres,7,1,-1,"{""coordinates"": [[[[-79.5266444706514, 43.7726..."


In [31]:
address_boundaries = address_boundaries.rename(columns={'PARCELID': 'parcel_id', 'ADDRESS_NUMBER': 'address_number', 'LINEAR_NAME_FULL': 'street_name'})
address_boundaries = address_boundaries[['parcel_id', 'address_number', 'street_name', 'geometry']]

address_boundaries['full_address'] = address_boundaries['address_number'].astype(str) + ' ' + address_boundaries['street_name']
address_boundaries = address_boundaries.drop(columns=['address_number', 'street_name'])
address_boundaries.head()
address_boundaries.to_csv("../data/processed/address_boundaries.csv", index=False)

In [33]:
address_boundaries = pd.read_csv("../data/processed/address_boundaries.csv")
address_boundaries.head()

,parcel_id,geometry,full_address
0,5090819,"{""coordinates"": [[[[-79.5230957815308, 43.7743...",5000 Jane St
1,5093788,"{""coordinates"": [[[[-79.525917669714, 43.77311...",343 Hullmar Dr
2,5124997,"{""coordinates"": [[[[-79.5579982280959, 43.7589...",40 Millwick Dr
3,5094544,"{""coordinates"": [[[[-79.5256074247922, 43.7724...",24 Wheelwright Cres
4,5095105,"{""coordinates"": [[[[-79.5266444706514, 43.7726...",36 Wheelwright Cres
